# 04 PSM 인과추론 - bureau 효과 분리

## 분석 질문
**관찰 사실:** `has_bureau` 그룹의 부도율이 더 높다.  
**의문:** 진짜로 bureau 보유가 부도를 높이는 인과인가, 아니면 bureau 보유자가 원래 다른 특성(고연령·소득 등)을 가져서인가?

## 접근 - Propensity Score Matching (PSM)
1. covariate(연령·소득·가족수·지역등급)로 bureau 보유 propensity score 추정 (로지스틱 회귀)
2. 보유군 1명마다 비보유군 1명을 propensity 가까운 사람으로 매칭 (nearest neighbor)
3. 매칭 후 covariate balance 확인 (SMD < 0.1 이상적)
4. 매칭된 두 그룹의 부도율 차이 = **ATT (Average Treatment Effect on Treated)**

## 산출물
- `outputs/figures/04_ps_distribution.png`
- `outputs/figures/04_psm_balance.png`
- `outputs/tables/04_psm_effect.csv`

**한계:** PSM은 관찰 가능한 confounder만 통제. 측정 안 된 confounder는 보정 불가.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

OUT_TBL = Path("../outputs/tables")
OUT_FIG = Path("../outputs/figures")
OUT_FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(OUT_TBL / "03_application_with_bureau.parquet")
df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25

cov_cols = ["AGE_YEARS", "AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS", "REGION_RATING_CLIENT"]
data = df[cov_cols + ["has_bureau", "TARGET"]].dropna()
print(f"PSM 분석 대상: {len(data):,}명")
print(f"has_bureau=1: {data['has_bureau'].sum():,}명 / has_bureau=0: {(data['has_bureau']==0).sum():,}명")

PSM 분석 대상: 307,509명
has_bureau=1: 263,490명 / has_bureau=0: 44,019명


In [ ]:
X = data[cov_cols]
T = data["has_bureau"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(X_scaled, T)
ps = ps_model.predict_proba(X_scaled)[:, 1]
data = data.copy()
data["ps"] = ps

fig, ax = plt.subplots(figsize=(8, 4))
data[data.has_bureau == 1]["ps"].hist(bins=50, alpha=0.5, label="has_bureau=1", ax=ax, color="steelblue")
data[data.has_bureau == 0]["ps"].hist(bins=50, alpha=0.5, label="has_bureau=0", ax=ax, color="coral")
ax.set_xlabel("Propensity Score")
ax.set_title("Common support 확인 - 두 그룹 PS 분포 겹침")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_FIG / "04_ps_distribution.png", dpi=100, bbox_inches="tight")
plt.show()

In [3]:
treated = data[data.has_bureau == 1].copy()
control = data[data.has_bureau == 0].copy()

nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[["ps"]])
distances, indices = nn.kneighbors(treated[["ps"]])

caliper = 0.05
mask = distances.flatten() <= caliper
matched_treated = treated[mask].copy()
matched_control = control.iloc[indices.flatten()[mask]].copy()
print(f"매칭 성공: {len(matched_treated):,}쌍 (treated {len(treated):,}명 중)")

매칭 성공: 263,490쌍 (treated 263,490명 중)


In [ ]:
def smd(t, c):
    pooled_std = np.sqrt((t.std()**2 + c.std()**2) / 2)
    return (t.mean() - c.mean()) / pooled_std

balance = pd.DataFrame({
    "covariate": cov_cols,
    "smd_before": [smd(treated[c], control[c]) for c in cov_cols],
    "smd_after":  [smd(matched_treated[c], matched_control[c]) for c in cov_cols],
})
print(balance.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(cov_cols))
ax.barh(x - 0.2, balance["smd_before"].abs(), 0.4, label="매칭 전", color="coral")
ax.barh(x + 0.2, balance["smd_after"].abs(),  0.4, label="매칭 후",  color="steelblue")
ax.set_yticks(x)
ax.set_yticklabels(cov_cols)
ax.axvline(0.1, color="red", linestyle="--", label="SMD 0.1 기준")
ax.set_title("Covariate Balance (SMD) - 매칭 전 vs 매칭 후")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_FIG / "04_psm_balance.png", dpi=100, bbox_inches="tight")
plt.show()

In [5]:
naive_diff = treated["TARGET"].mean() - control["TARGET"].mean()
y1 = matched_treated["TARGET"].mean()
y0 = matched_control["TARGET"].mean()
att = y1 - y0

np.random.seed(42)
boot_atts = []
n = len(matched_treated)
for _ in range(1000):
    idx = np.random.choice(n, n, replace=True)
    boot_atts.append(
        matched_treated.iloc[idx]["TARGET"].mean() - matched_control.iloc[idx]["TARGET"].mean()
    )
ci_low, ci_high = np.percentile(boot_atts, [2.5, 97.5])

print(f"Naive 차이 (매칭 전): {naive_diff*100:+.2f}%p")
print(f"ATT  (매칭 후 인과): {att*100:+.2f}%p  [95% CI: {ci_low*100:.2f}, {ci_high*100:.2f}]")

result = pd.DataFrame([{
    "naive_diff_pct": round(naive_diff*100, 4),
    "att_pct":        round(att*100, 4),
    "att_ci_low":     round(ci_low*100, 4),
    "att_ci_high":    round(ci_high*100, 4),
    "matched_pairs":  len(matched_treated),
}])
result.to_csv(OUT_TBL / "04_psm_effect.csv", index=False)
result

Naive 차이 (매칭 전): -2.40%p
ATT  (매칭 후 인과): -2.14%p  [95% CI: -2.30, -1.99]


,naive_diff_pct,att_pct,att_ci_low,att_ci_high,matched_pairs
0,-2.3951,-2.142,-2.2953,-1.9936,263490


## 결론

**Naive 차이 (관찰):** has_bureau=1 부도율이 비보유보다 낮음 (-2.40%p)  
**ATT (인과):** 매칭 후 차이 -2.14%p [CI: -2.30, -1.99]

**해석:**
- |ATT| < |Naive| → 관찰 차이의 일부는 bureau 보유 자체가 아니라 나이·소득 등 다른 특성 차이에서 옴
- ATT 신뢰구간이 0을 포함하지 않음 → bureau 기록 없음 자체가 인과적으로 부도 위험 신호

**한계:**
- 측정 안 된 confounder (재정 관리 습관 등) 통제 불가
- caliper 0.05 선택에 robustness 검증 미실시
- 1:1 매칭이라 비매칭된 treated 일부 손실